In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Project 4 - Databricks Streaming Producer (simulated live telematics)
# MAGIC
# MAGIC Generates and sends simulated live telematics events to Event Hub `eh-telematics`,
# MAGIC grounded in the same real per-PID stats (mean/stddev/min/max, alarm-class value
# MAGIC bounds) pulled from `silver_telematics_events` that the batch synthetic fleet used -
# MAGIC same statistical grounding, different purpose: this is fresh simulated live events
# MAGIC for the streaming leg, not a replay of rows already loaded through the batch
# MAGIC pipeline (per the locked design decision - telematics devices don't originate from
# MAGIC a database, so streaming stays a simulated event producer into Event Hub).
# MAGIC
# MAGIC Scaled to ~75,000 events for the pilot (10 simulated devices x 150 readings x 53
# MAGIC PIDs = 79,500), matching the project's locked streaming pilot scale - much smaller
# MAGIC than the 500-vehicle/3.975M-row batch synthetic fleet, since this represents a live
# MAGIC feed window, not a historical backfill.
# MAGIC
# MAGIC **One-time setup before running:**
# MAGIC 1. `%pip install azure-eventhub` (not the Spark connector - this notebook is a
# MAGIC    plain Python producer; sends don't need Spark at all, only the stats pull does).
# MAGIC 2. Needs `silver_telematics_events` already built (run after silver, same as generate).
# MAGIC 3. Reads the Event Hub connection string from the `project4-eventhub` secret scope
# MAGIC    (Key Vault-backed, pointing at `kv-project4`'s `eventhub-connection-string`
# MAGIC    secret). This needs **Send** rights on the Event Hub - confirm the underlying
# MAGIC    SAS policy isn't Listen-only or the send will fail partway through with an
# MAGIC    unauthorized error.
# MAGIC 4. Run this BEFORE `nb_streaming_bronze_databricks` - there's nothing for the
# MAGIC    consumer to read until this has sent events.

In [0]:
%pip install azure-eventhub

In [0]:
dbutils.widgets.text("catalog", "policybench_dev")
catalog = dbutils.widgets.get("catalog")
spark.sql(f"USE CATALOG {catalog}")

In [0]:
SECRET_SCOPE = "project4-eventhub"
CONNECTION_STRING_SECRET = "eventhub-connection-string"
EVENT_HUB_NAME = "eh-telematics"
 
N_DEVICES = 10
READINGS_PER_DEVICE = 150
DEVICE_PREFIX = "LIVE"

In [0]:
import json
import random
import time
from datetime import datetime, timezone
from azure.eventhub import EventHubProducerClient, EventData, TransportType
from pyspark.sql import functions as F
 
 
def log_pipeline_run(spark, platform, layer, start_dt, end_dt):
    duration = round((end_dt - start_dt).total_seconds(), 2)
    log_row = spark.createDataFrame([{
        "platform": platform, "layer": layer,
        "start_ts": start_dt, "end_ts": end_dt, "duration_seconds": duration,
    }])
    log_row.write.format("delta").mode("append").saveAsTable("pipeline_run_log")
    print(f"[{platform}/{layer}] duration: {duration}s")

In [0]:
# MAGIC %md
# MAGIC ## Pull real per-PID stats and alarm-class bounds
# MAGIC Same source and same logic as the batch fleet generator (`generate_synthetic_fleet`
# MAGIC in `nb_generate_databricks.py`) - deliberately reused rather than reimplemented, so
# MAGIC both the batch synthetic fleet and this streaming feed are grounded in the same
# MAGIC real-world value ranges and alarm thresholds.

In [0]:
silver_telematics = spark.read.table("silver_telematics_events")
clean_real = silver_telematics.filter(~F.col("VALUE_OUTLIER"))

In [0]:
pid_stats_df = clean_real.groupBy("PID").agg(
    F.mean("value").alias("mean_val"),
    F.stddev("value").alias("std_val"),
    F.min("value").alias("min_val"),
    F.max("value").alias("max_val"),
).fillna({"std_val": 0.01})

In [0]:
alarm_bounds_df = (
    clean_real.filter(F.col("alarm_class").isNotNull())
    .groupBy("PID", "alarm_class")
    .agg(F.min("value").alias("class_min"), F.max("value").alias("class_max"))
)

In [0]:
pid_stats = {row["PID"]: row.asDict() for row in pid_stats_df.collect()}
alarm_bounds = {}
for row in alarm_bounds_df.collect():
    alarm_bounds.setdefault(row["PID"], []).append(
        (row["alarm_class"], row["class_min"], row["class_max"])
    )

In [0]:
pids = list(pid_stats.keys())
print(f"Loaded stats for {len(pids)} PIDs")

In [0]:
# MAGIC %md
# MAGIC ## Generate simulated live events
# MAGIC Plain Python (not Spark) - this producer sends one event at a time, batched into
# MAGIC Event Hub sends, not a distributed transform, so plain `random`/Python loops are
# MAGIC the right tool here rather than a Spark DataFrame.

In [0]:
def classify_alarm(pid, value):
    for alarm_class, class_min, class_max in alarm_bounds.get(pid, []):
        if class_min <= value <= class_max:
            return alarm_class
    return 0

In [0]:
def generate_event(device_id, pid):
    stats = pid_stats[pid]
    raw_value = random.gauss(stats["mean_val"], stats["std_val"])
    value = round(min(stats["max_val"], max(stats["min_val"], raw_value)), 3)
    return {
        "device_id": device_id,
        "timestamp": int(time.time() * 1000),
        "PID": pid,
        "value": value,
        "alarm_class": classify_alarm(pid, value),
    }

In [0]:
devices = [f"{DEVICE_PREFIX}-{i:04d}" for i in range(1, N_DEVICES + 1)]
events = [
    generate_event(device_id, pid)
    for device_id in devices
    for _ in range(READINGS_PER_DEVICE)
    for pid in pids
]

In [0]:
print(f"Generated {len(events)} events for {len(devices)} devices x {READINGS_PER_DEVICE} readings x {len(pids)} PIDs")

In [0]:
# MAGIC %md
# MAGIC ## Send to Event Hub
# MAGIC Batches events into `EventDataBatch` objects (respects the Event Hub's max batch
# MAGIC size automatically - `add()` raises `ValueError` when a batch is full, caught here
# MAGIC to flush and start a new batch) rather than one network call per event.

In [0]:
start_dt = datetime.now(timezone.utc)
 
connection_str = dbutils.secrets.get(scope=SECRET_SCOPE, key=CONNECTION_STRING_SECRET)
producer = EventHubProducerClient.from_connection_string(
    conn_str=connection_str,
    eventhub_name=EVENT_HUB_NAME,
    transport_type=TransportType.AmqpOverWebsocket,
)
 
sent_count = 0
with producer:
    batch = producer.create_batch()
    for event in events:
        payload = json.dumps(event)
        try:
            batch.add(EventData(payload))
        except ValueError:
            producer.send_batch(batch)
            sent_count += len(batch)
            batch = producer.create_batch()
            batch.add(EventData(payload))
    if len(batch) > 0:
        producer.send_batch(batch)
        sent_count += len(batch)
 
end_dt = datetime.now(timezone.utc)
log_pipeline_run(spark, "Databricks", "streaming_producer", start_dt, end_dt)
 
print(f"Sent {sent_count} events to {EVENT_HUB_NAME}")

## Sanity check

In [0]:
assert sent_count == len(events), (
    f"Sent {sent_count} but generated {len(events)} - some events may not have reached "
    "Event Hub. Check for errors above before running the bronze streaming consumer."
)